# 12.15 幾種真人語音問題，如何從零學會回應？


有人說「我的銀行 App 打不開」，我們希望助手回一個短提示，例如「先檢查網路，再重新啟動 App。」這裡想教的是聽懂幾種需求後回應，不要求把任何人說的每個字都完整轉寫下來。

適合這個方向的公開材料是 MInDS-14 的中文真人錄音。它收集少數銀行情境的口語需求：參與者看到意圖說明與例句，再自己說出新的問法。**意圖**（intent）在這裡就是需求種類，例如更新地址、App 出問題、卡片無法使用。本節用這三種來設計短而一般的回覆；本節設計的助理沒有連接銀行帳戶，不能回報未知餘額或完成帳戶操作。

聲音怎麼接到文字回答？先沿用前面學過的 log-mel 特徵：把錄音切成時間框，算出各 mel 頻帶的功率，再做固定的 log 轉換，得到按時間排列的一列數字。這一步整理聲音的變化，沒有寫出「App」兩個字，也沒有決定應回哪一句話。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/new-12.15-direct-audio.svg")))

接著，**語音入口**（speech encoder）用可學的參數，從這列聲音數字算出需求線索；它要從真人說話學出需求種類，而不是單音的高低。接頭把線索轉成文字核心能接收的特徵位置，也就是放進模型序列中的一組數字表示。核心讀到這些位置與對話歷史，生成中文答覆。

語音入口和 12.8 的聲音入口都從逐框的 log-mel 出發，但不是一個時間框換一條特徵。真人錄音長短不一，本課實際訓練的語音入口（[tiny_perceptron/selftrained/model.py](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/tiny_perceptron/selftrained/model.py) 的 `AudioEncoder`，和 12.8 那個同名類別不是同一份程式）分三步整理時間：先把相鄰幾框合起來算、每次往後跳 2 框，連做兩層，序列約縮成四分之一長；再沿時間正反各讀一遍，讓每個位置都參考前後的聲音；最後沿時間順序切出 16 個區間，各自取平均（`adaptive_avg_pool1d`；區間邊界取整，相鄰區間可能重疊）。不論錄音多長，它都交出 16 條有先後順序的特徵，所以聲音特徵在模型序列中固定佔 16 個位置，不是有幾個時間框就佔幾個。這個 16 是時間上切出的段數，和 12.6 的 16 個 mel 頻帶無關；第 19 章的 MoE 助理用的就是這個入口，見 [19.6](https://birdhackor.github.io/tiny-perceptron-vlm/19.6.html)。

下面只看形狀：用隨機初值建立這個語音入口，再用隨機數代替三段錄音的 log-mel，分別有 173、600、1000 個時間框，每框 40 個 mel 頻帶（這個入口只收 40 帶）。


In [ ]:
import torch
from tiny_perceptron.selftrained.model import AudioEncoder

torch.manual_seed(0)
encoder = AudioEncoder(64)
for frames in [173, 600, 1000]:
    mel = torch.randn(frames, 40)
    shortened = encoder.temporal(mel.T[None]).shape[-1]
    features, _ = encoder(mel)
    print("輸入", tuple(mel.shape), "第一步後", shortened, "輸出", tuple(features.shape))

`encoder.temporal` 是第一步的兩層，`mel.T[None]` 把特徵排成它要的「筆數、頻帶、時間框」；印出的 44、150、250 是第一步後的時間長度，約為原本的四分之一。`encoder(mel)` 跑完三步，三段的第一個回傳值都是 `(16,64)`，也就是 16 條特徵、每條 64 個數字；64 是建立時指定的特徵寬度，第 19 章的 MoE 助理用 256。第二個回傳值是三種需求各自的分數，本節用不到，用 `_` 接住。

若要從零建立這條路，語音入口、接頭與文字核心的參數都從隨機初值開始，並需要各自的訓練安排。分階段訓練並不違反「從零」：12.9 的單音實驗也是先單獨訓練聲音入口，再接上文字核心一起訓練。固定的 log-mel 計算可以使用一般工具，因為它不是已經學會中文的神經模型。辨識需求與產生答覆的部分，則由這些自行訓練的元件處理。

訓練材料需要把錄音與這個任務的正確短答配在一起。MInDS-14 提供需求種類，沒有提供我們要教的客服答覆，因此答案要另外撰寫、核對。訓練可以分兩步。第一步用打字版的問法（例如作者為每種需求寫的幾句文字）配上這些答案，讓文字核心先學會回應三種需求。第二步再訓練語音入口與接頭，讓真人錄音送進核心後，引出和打字版相同的回應。分兩步的好處是出錯時查得出在哪一段：打字版也答錯，先查文字核心；打字版答對、語音版答錯，先查語音入口與接頭，也要確認核心能用語音送來的特徵。訓練時用正確答案和預測比較、調整參數；推論時只送入錄音與對話，不把正確答案一起送進去。

要看是否使用了聲音，可以做對照：保留相同背景與提問方式，只改用另一類需求的錄音，檢查回覆主題是否跟著改變。還要把一部分原始錄音與新問法留作測試資料，不拿來更新參數。同一段聲音的裁切或加噪版本仍屬同一家族（由同一段原始錄音衍生），不能分散到訓練與測試兩邊，再算成陌生錄音。如果所有問題都得到「請聯絡客服」，雖然語句合理，也不足以說明模型分清了這三種需求。

這和 12.14 的語音辨識路線有不同目標。ASR 要回答「他說了哪些字」；此處直接問答要回答「這個已教過的需求，該怎麼回應」。沒有產生轉寫文字，不能稱作通用 ASR；能分清三種需求，也不推出能聽懂新數字、物件名或任意中文問題。

聲音特徵經過自行訓練的語音入口，可以成為同一文字核心的另一種輸入。下一節只增加對話歷史，看看文字與聲音如何接續同一段需求。

<details>
<summary>公開材料、方法與例子的性質</summary>

PolyAI 的[MInDS-14 官方資料卡](https://huggingface.co/datasets/PolyAI/minds14/blob/40ce77cb32a384e4d50a568e1ec39ac804019d33/README.md)標示 CC BY 4.0，中文有 502 條，各類可用短句仍需核對；沒有公開說話者 ID，不能據此宣稱跨說話者驗收。[原論文](https://arxiv.org/abs/2104.08524)說明真人口語收集，也說明其實驗使用現成 ASR 與預訓練文字編碼器。本節借用的是合法來源與有限需求設計，不引用那些成績當從零訓練的證據。

本文沒有讀取、播放或轉寫某段公開錄音。問句、希望答案與聲音色塊都是作者設計示例。直接語音需求辨識可參考[端到端語音理解研究](https://arxiv.org/abs/1904.03670)，但該研究涉及預訓練，不能拿其結果保證本課隨機初值模型的能力。

</details>
